# Day 1: Decision Trees

## Overview

Decision trees are the foundation of all ensemble methods. Learn how trees grow via information gain, control overfitting with hyperparameters, and extract feature importance for model debugging.

## Learning Objectives

- Understand Gini impurity and entropy as split criteria
- Learn how trees prevent overfitting (max_depth, min_samples_leaf)
- Extract and interpret feature importance
- Compare classification trees vs regression trees

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Understanding Gini Impurity and Tree Growth

In [ ]:
# Load dataset
iris = load_iris()
X = iris.data[:, :2]  # Use first 2 features for visualization
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Gini impurity calculation
def gini_impurity(y):
    """Calculate Gini impurity for a node."""
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    gini = 1 - np.sum(probabilities ** 2)
    return gini

# Test on root node
root_gini = gini_impurity(y_train)
print(f"Root node Gini: {root_gini:.4f}")
print(f"Class distribution: {np.unique(y_train, return_counts=True)}")

## 2. Training Decision Trees with Different Depths

In [ ]:
# Train trees with different max_depth
depths = [1, 3, 5, 10, None]
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_pred = dt.predict(X_train)
    test_pred = dt.predict(X_test)
    train_scores.append(accuracy_score(y_train, train_pred))
    test_scores.append(accuracy_score(y_test, test_pred))

# Plot bias-variance trade-off
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(range(len(depths)), train_scores, marker='o', label='Train Score', linewidth=2)
ax.plot(range(len(depths)), test_scores, marker='s', label='Test Score', linewidth=2)
ax.set_xticks(range(len(depths)))
ax.set_xticklabels([str(d) if d else 'None' for d in depths])
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Bias-Variance Trade-off')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Find best depth
best_depth_idx = np.argmax(test_scores)
best_depth = depths[best_depth_idx]
print(f"Best depth: {best_depth} (test accuracy: {test_scores[best_depth_idx]:.4f})")

## 3. Visualizing a Decision Tree

In [ ]:
# Train and visualize a shallow tree
dt_shallow = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_shallow.fit(X_train, y_train)

# Plot the tree
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_shallow, feature_names=iris.feature_names[:2], class_names=iris.target_names,
          filled=True, ax=ax, fontsize=10)
plt.title('Decision Tree (max_depth=3)')
plt.tight_layout()
plt.show()

print(f"Tree depth: {dt_shallow.get_depth()}")
print(f"Number of leaves: {dt_shallow.get_n_leaves()}")

## 4. Feature Importance

In [ ]:
# Train on full Iris dataset
iris_X = iris.data
iris_y = iris.target
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    iris_X, iris_y, test_size=0.3, random_state=42
)

dt_full = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_full.fit(X_train_full, y_train_full)

# Extract feature importance
feature_importance = dt_full.feature_importances_
sorted_idx = np.argsort(feature_importance)[::-1]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(sorted_idx)), feature_importance[sorted_idx])
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([iris.feature_names[i] for i in sorted_idx])
ax.set_xlabel('Importance')
ax.set_title('Feature Importance in Decision Tree')
plt.tight_layout()
plt.show()

# Accuracy
train_acc = accuracy_score(y_train_full, dt_full.predict(X_train_full))
test_acc = accuracy_score(y_test_full, dt_full.predict(X_test_full))
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

## 5. Hyperparameter Tuning: min_samples_leaf and min_samples_split

In [ ]:
# Test different min_samples_leaf values
min_leaf_values = [1, 2, 5, 10, 20, 50]
train_scores_leaf = []
test_scores_leaf = []

for min_leaf in min_leaf_values:
    dt = DecisionTreeClassifier(max_depth=10, min_samples_leaf=min_leaf, random_state=42)
    dt.fit(X_train_full, y_train_full)
    train_pred = dt.predict(X_train_full)
    test_pred = dt.predict(X_test_full)
    train_scores_leaf.append(accuracy_score(y_train_full, train_pred))
    test_scores_leaf.append(accuracy_score(y_test_full, test_pred))

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(min_leaf_values, train_scores_leaf, marker='o', label='Train', linewidth=2)
ax.plot(min_leaf_values, test_scores_leaf, marker='s', label='Test', linewidth=2)
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('Accuracy')
ax.set_title('Effect of min_samples_leaf on Tree Performance')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal min_samples_leaf: {min_leaf_values[np.argmax(test_scores_leaf)]}")

## Exercises

1. Train a decision tree on the breast cancer dataset. Plot train vs test accuracy as a function of max_depth.
2. Compare Gini criterion vs entropy criterion. Which gives better accuracy on Iris?
3. Tune max_depth and min_samples_leaf using GridSearchCV.

## Solutions

See completed examples in the cells above. Key takeaways:
- Shallow trees underfit; deep trees overfit
- Use cross-validation to find the sweet spot
- Feature importance helps debug your model